Análise de Filtros para Imagens de Raio-X

Este documento apresenta uma análise técnica dos métodos de processamento de imagem aplicados ao notebook, focando em detecção de bordas e segmentação.


1. Carregamento e Preparação dos Dados

O código utiliza a biblioteca imageio para leitura de metadados e pixels. No processamento inicial, observa-se a tentativa de carregar sequências de imagens para criar uma representação volumétrica ou temporal (GIF), o que é comum em exames de diagnóstico por imagem para avaliar diferentes "fatias" ou momentos do exame.

In [ ]:
import os
import imageio

DIR = "tutorial-x-ray-image-processing"

xray_image = imageio.v3.imread(os.path.join(DIR, "00000011_001.png"))

FileNotFoundError: No such file: '/home/guilhermemendes/Área de trabalho/MetodosNumericos/tutorial-x-ray-image-processing/00000011_001.png'

In [ ]:
print(xray_image.shape)
print(xray_image.dtype)

In [ ]:
import matplotlib.pyplot as plt

fig, ax = plt.subplots()
ax.imshow(xray_image, cmap="gray")
ax.set_axis_off()

In [ ]:
import numpy as np
num_imgs = 9

combined_xray_images_1 = np.array(
    [imageio.v3.imread(os.path.join(DIR, f"00000011_00{i}.png")) for i in range(num_imgs)]
)

In [ ]:
combined_xray_images_1.shape

2. Filtros de Detecção de Bordas (Métodos Derivativos)

A detecção de bordas baseia-se na identificação de variações bruscas de intensidade de pixels. No notebook, foram aplicados os seguintes métodos:

A. Laplaciano de Gaussiana (LoG)

Implementado via ndimage.gaussian_laplace. Este método combina dois passos:
Suavização Gaussiana: Reduz o ruído da imagem (que pode causar falsas bordas).
Operador Laplaciano: Uma derivada de segunda ordem que identifica regiões onde a taxa de variação da intensidade muda de sinal (cruzamento por zero).
Aplicação: É excelente para encontrar o contorno exato de estruturas anatômicas densas no raio-X.

B. Magnitude do Gradiente Gaussiano

Utiliza ndimage.gaussian_gradient_magnitude. Ao contrário do Laplaciano, este utiliza a primeira derivada.
Ele calcula a inclinação (gradiente) da intensidade da imagem após a suavização.
O resultado destaca áreas onde há uma transição de "claro para escuro", resultando em bordas mais espessas e contínuas.

C. Operador Sobel

O filtro Sobel é um dos mais clássicos na computação visual. Ele calcula a aproximação do gradiente da função de intensidade da imagem.
O código aplica o filtro nos eixos X (axis=0) e Y (axis=1).
A magnitude final é calculada via np.hypot(x_sobel, y_sobel), que combina os gradientes horizontais e verticais.
Vantagem: É muito eficaz para destacar linhas verticais e horizontais específicas, como as costelas ou a coluna vertebral em um raio-X de tórax.

D. Abordagem Canny / Prewitt

Embora o algoritmo de Canny completo envolva várias etapas (suavização, gradiente, supressão de não-máximos e limiarização por histerese), o notebook aplica uma variação usando:
Filtro de Fourier Gaussiano: Para suavização no domínio da frequência.
Operador Prewitt: Similar ao Sobel, mas utiliza pesos diferentes em sua matriz (kernel) para detectar bordas.



In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=num_imgs, figsize=(30, 30))

for img, ax in zip(combined_xray_images_1, axes):
    ax.imshow(img, cmap='gray')
    ax.axis('off')

In [ ]:
GIF_PATH = os.path.join(DIR, "xray_image.gif")
imageio.mimwrite(GIF_PATH, combined_xray_images_1, format= ".gif", duration=1000)

In [ ]:
from scipy import ndimage

xray_image_laplace_gaussian = ndimage.gaussian_laplace(xray_image, sigma=1)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Laplacian-Gaussian (edges)")
axes[1].imshow(xray_image_laplace_gaussian, cmap="gray")

for i in axes:
    i.axis("off")

In [ ]:
x_ray_image_gaussian_gradient = ndimage.gaussian_gradient_magnitude(xray_image, sigma=2)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=2, figsize=(10, 10))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Gaussian gradient (edges)")
axes[1].imshow(x_ray_image_gaussian_gradient, cmap="gray")
for i in axes:
    i.axis("off")

In [ ]:
x_sobel = ndimage.sobel(xray_image, axis=0)
y_sobel = ndimage.sobel(xray_image, axis=1)

xray_image_sobel = np.hypot(x_sobel, y_sobel)

xray_image_sobel *= 255.0 / np.max(xray_image_sobel)

In [ ]:
print("The data type - before: ", xray_image_sobel.dtype)

xray_image_sobel = xray_image_sobel.astype("float32")

print("The data type - after: ", xray_image_sobel.dtype)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=3, figsize=(15, 15))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Sobel (edges) - grayscale")
axes[1].imshow(xray_image_sobel, cmap="gray")
axes[2].set_title("Sobel (edges) - CMRmap")
axes[2].imshow(xray_image_sobel, cmap="CMRmap")
for i in axes:
    i.axis("off")

In [ ]:
fourier_gaussian = ndimage.fourier_gaussian(xray_image, sigma=0.05)

x_prewitt = ndimage.prewitt(fourier_gaussian, axis=0)
y_prewitt = ndimage.prewitt(fourier_gaussian, axis=1)

xray_image_canny = np.hypot(x_prewitt, y_prewitt)

xray_image_canny *= 255.0 / np.max(xray_image_canny)

print("The data type - ", xray_image_canny.dtype)

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=4, figsize=(20, 15))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Canny (edges) - prism")
axes[1].imshow(xray_image_canny, cmap="prism")
axes[2].set_title("Canny (edges) - nipy_spectral")
axes[2].imshow(xray_image_canny, cmap="nipy_spectral")
axes[3].set_title("Canny (edges) - terrain")
axes[3].imshow(xray_image_canny, cmap="terrain")
for i in axes:
    i.axis("off")

In [ ]:
print("The data type of the X-ray image is: ", xray_image.dtype)
print("The minimum pixel value is: ", np.min(xray_image))
print("The maximum pixel value is: ", np.max(xray_image))
print("The average pixel value is: ", np.mean(xray_image))
print("The median pixel value is: ", np.median(xray_image))

In [ ]:
pixel_intensity_distribution = ndimage.histogram(
    xray_image, min=np.min(xray_image), max=np.max(xray_image), bins=256
)

fig, ax = plt.subplots()
ax.plot(pixel_intensity_distribution)
ax.set_title("Pixel intensity distribution")

In [ ]:
# The threshold is "greater than 150"
# Return the original image if true, `0` otherwise
xray_image_mask_noisy = np.where(xray_image > 150, xray_image, 0)

fig, ax = plt.subplots()
ax.imshow(xray_image_mask_noisy, cmap="gray")
ax.set_axis_off()

In [ ]:
# The threshold is "greater than 150"
# Return `1` if true, `0` otherwise
xray_image_mask_less_noisy = np.where(xray_image > 150, 1, 0)

fig, ax = plt.subplots()
ax.imshow(xray_image_mask_less_noisy, cmap="gray")
ax.set_axis_off()

In [ ]:
fig, axes = plt.subplots(nrows=1, ncols=9, figsize=(30, 30))

axes[0].set_title("Original")
axes[0].imshow(xray_image, cmap="gray")
axes[1].set_title("Laplace-Gaussian (edges)")
axes[1].imshow(xray_image_laplace_gaussian, cmap="gray")
axes[2].set_title("Gaussian gradient (edges)")
axes[2].imshow(x_ray_image_gaussian_gradient, cmap="gray")
axes[3].set_title("Sobel (edges) - grayscale")
axes[3].imshow(xray_image_sobel, cmap="gray")
axes[4].set_title("Sobel (edges) - hot")
axes[4].imshow(xray_image_sobel, cmap="hot")
axes[5].set_title("Canny (edges) - prism)")
axes[5].imshow(xray_image_canny, cmap="prism")
axes[6].set_title("Canny (edges) - nipy_spectral)")
axes[6].imshow(xray_image_canny, cmap="nipy_spectral")
axes[7].set_title("Mask (> 150, noisy)")
axes[7].imshow(xray_image_mask_noisy, cmap="gray")
axes[8].set_title("Mask (> 150, less noisy)")
axes[8].imshow(xray_image_mask_less_noisy, cmap="gray")
for i in axes:
    i.axis("off")

3. Conclusão

O processamento demonstra um fluxo completo: desde a remoção de ruído com filtros Gaussianos até a extração de características estruturais com Sobel e Laplaciano. O uso de diferentes mapas de cores (colormaps) como 'gray', 'hot' e 'nipy_spectral' auxilia na visualização humana de detalhes que poderiam passar despercebidos em uma escala de cinza padrão.